## sinter_sampler.py

In [ ]:
import stim
import sinter
from corrqec2.sampling import SinterSampler

# Experiment
experiment_name = "SurfaceCodeMemory"
experiment_args = {'distance': 9, 'rounds': '3d', 'basis': 'Z'}  # rounds can be multiples of the distance e.g., 3*d


# Noise model
noise_model_name = "StormModel"
# These tune the independent gate level noise that we let Stim handle
# (separate from all the correlated noise that StormModel injects on top)
p_gate = 0.01
gate_noise = {
    #'after_identity_depolarization': p_gate,
    'after_clifford_depolarization': p_gate,
    'before_measure_flip_probability': p_gate,
    #'after_reset_flip_probability': p_gate,
}
# These tune the correlated noise that the StormModel will generate
noise_model_params = {
        'a': 0.01,
        'b': 0.5,
        'emissions': [[1., 0., 0., 0.], [0.25, 0.25, 0.25, 0.25]],
    }
noisy_qubit_types = 'syndrome'  # controls which qubits get the correlated noise
# Putting it all together
noise_model_args = {
    'model_params': noise_model_params,
    'gate_noise': gate_noise,
    'noisy_qubit_types': noisy_qubit_types,
}

# Decoder
decoder_name = "Pymatching"
marginalized_dem = True

min_batch_size = 1000

metadata = {
    "experiment": experiment_name,
    "experiment_args": experiment_args,
    "noise_model": noise_model_name,
    "noise_model_args": noise_model_args,
    "decoder": decoder_name,
    "min_batch_size": min_batch_size,
    "marginalized_detector_error_model": marginalized_dem,
}

task = sinter.Task(circuit=stim.Circuit(), json_metadata=metadata)
sampler = SinterSampler()

In [ ]:
num_workers = 4
n_shots = 10000
print_progress = True

stats = sinter.collect(tasks=[task], num_workers=num_workers, decoders='custom_sampler', custom_decoders={'custom_sampler': sampler}, max_shots=n_shots, print_progress=print_progress)

In [ ]:
def collected_stats_to_csv(stats: sinter.TaskStats, filepath: str):
    with open(filepath, 'w') as f:
        print(sinter.CSV_HEADER, file=f)
        for stat in stats:
            print(stat.to_csv_line(), file=f)

collected_stats_to_csv(stats, "task1.csv")

In [ ]:
from pathlib import Path
import numpy as np
from corrqec2.sampling import create_task, run_tasks_to_csv, get_default_parser

def calc_a_b(p_bar, Delta):
    """Calculate storm model parameters a, b for given spectral gap Delta (=a+b) and fixed marginal error rate p_bar."""
    a = 4 * p_bar * Delta / 3
    b = Delta * (1 - 4 * p_bar / 3)
    return a, b


def calc_Delta_from_xi(xi):
    return 1 - np.exp(-1 / xi)


def calc_a_b_from_xi(p_bar, xi):
    Delta = calc_Delta_from_xi(xi)
    return calc_a_b(p_bar, Delta)



# Parse command-line arguments
# parser = get_default_parser()
# args = parser.parse_args()

# Configure output path
# output_dir = Path.home() / "mx95_scratch2" / "jkam" / "corrqec2_results"

# Sweep over distances and correlation lengths
tasks = []
xis = [2, 4, 6, 8, 12, 16, 20, 28]
distances = [5, 7, 9, 11, 13, 15, 17, 19]

# Fixed noise model parameters
p_gate = 0.001
gate_noise = {
    # "after_identity_depolarization": p_gate,
    "after_clifford_depolarization": p_gate,
    "before_measure_flip_probability": p_gate,
    "after_reset_flip_probability": p_gate,
}

for distance in distances:
    for xi in xis:
        a, b = calc_a_b_from_xi(p_bar=p_gate, xi=xi)

        task = create_task(
            experiment="SurfaceCodeMemory",
            experiment_args={"distance": distance, "rounds": "3d", "basis": "Z"},
            noise_model="StormModel",
            noise_model_args={
                "model_params": {
                    "a": a,
                    "b": b,
                    "emissions": [[1.0, 0.0, 0.0, 0.0], [0.25, 0.25, 0.25, 0.25]],
                },
                "gate_noise": gate_noise,
                "noisy_qubit_types": "data",
            },
            decoder="Pymatching",
            marginalized_detector_error_model=True,
        )
        tasks.append(task)

In [ ]:
from corrqec2.experiments import SurfaceCodeMemory
from corrqec2.noisemodels import StormModel
from corrqec2.decoding import Pymatching
from corrqec2.sampling import Sampler

distance = 3

experiment=SurfaceCodeMemory
experiment_args={"distance": distance, "rounds": "3d", "basis": "Z"}
noise_model=StormModel
noise_model_args={
                "model_params": {
                    "a": a,
                    "b": b,
                    "emissions": [[1.0, 0.0, 0.0, 0.0], [0.25, 0.25, 0.25, 0.25]],
                },
                "gate_noise": gate_noise,
                "noisy_qubit_types": "data",
            }

experiment = SurfaceCodeMemory(**experiment_args)
noise_model = StormModel(**noise_model_args)

decoder = Pymatching()

sampler = Sampler(experiment, noise_model, decoder, marginalized_detector_error_model=True)

In [ ]:
circ = sampler.noise_model.gen_marginalized_circuit(experiment)

In [ ]:
circ.diagram('timeline-svg')